In [1]:
import yaml
import torch
import pytorch_lightning as pl
from tqdm.auto import tqdm
import torch.nn as nn
import numpy as np 

/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [26]:
import h5py
import numpy as np
import os
import pandas as pd
import pathlib
import pickle
import scipy.stats as stats
import soxr
#! change below to spatial_attn_lighting if want to use with modular 
import src.spatial_attn_lightning as attn_tracking_lightning
import src.audio_transforms as at
import torch
import yaml

import argparse
from argparse import ArgumentParser
from corpus.speaker_room_dataset import SpeakerRoomDataset
from tqdm.auto import tqdm
from datetime import datetime
import sys 
import torchaudio.transforms as T

sys.path.append('../')
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

In [3]:
config_path = "config/binaural_attn/word_task_half_co_loc_v07.yaml"
ckpt_path = "attn_cue_models/word_task_half_co_loc_v07/checkpoints/epoch=2-step=46074.ckpt"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

model = attn_tracking_lightning.BinauralAttentionModule.load_from_checkpoint(checkpoint_path=ckpt_path, config=config, strict=False).cuda()


num_classes={'num_words': 800}
Model performing word task


/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/torchaudio/functional/functional.py:1371: UserWarning: "kaiser_window" resampling method name is being deprecated and replaced by "sinc_interp_kaiser" in the next release. The default behavior remains unchanged.
  warnings.warn(


center_crop=True
binaural=True
Binaural cochleagram
using FIR cochleagram


In [32]:
model_name = pathlib.Path(config_path).stem


In [35]:
from corpus.jsinV3_attn_tracking_multi_talker_background import jsinV3_attn_tracking_multi_talker_background
import src.audio_transforms as at

coch_gram = model.coch_gram.cuda()
label_type = 'CV'
sr = config['audio']['rep_kwargs']['sr']
config['data'] = {}
config['data']['corpus'] = {}
config['data']['corpus']['n_talkers'] = 1
config['data']['corpus']['root'] = '/om2/user/msaddler/projects/ibmHearingAid/assets/data/datasets/JSIN_v3.00/nStim_20000/2000ms/rms_0.1/noiseSNR_-10_10/stimSR_20000/reverb_none/noise_all/JSIN_all_v3/subsets/' # Path to raw GigaSpeech dataset


snr=0
audio_transforms = at.AudioCompose([
                at.AudioToTensor(),
                at.CombineWithRandomDBSNR(low_snr=-snr, high_snr=snr), 
                at.RMSNormalizeForegroundAndBackground(rms_level=0.02),  # 0.02 is the default for CV-based models 
                at.UnsqueezeAudio(dim=0),
                at.DuplicateChannel() # only need to copy channels here
        ])  



# def upsample op
upsample = T.Resample(20_000, sr,
                    lowpass_filter_width=64,
                    rolloff=0.9475937167399596,
                    beta=14.769656459379492,
                    resampling_method='kaiser_window',
                    dtype=torch.float32)
                            
# get dataset
dataset = jsinV3_attn_tracking_multi_talker_background(**config['data']['corpus'],
                                            mode='val',
                                            transform=None,
                                            demo=True)

def collate_fn(batch):
    #apply transforsms to batch
    cues = []
    foregrounds = []
    backgrounds = []
    mixtures = []
    labels = []
    for (fg, bg, cue, label) in batch:
        cue = audio_transforms(upsample(torch.from_numpy(cue)).squeeze(), None)[0]
        cues.append(cue)
        fg = upsample(torch.from_numpy(fg)).squeeze()
        bg = upsample(torch.from_numpy(bg)).squeeze()
        foregrounds.append(audio_transforms(fg, None)[0])
        backgrounds.append(audio_transforms(bg, None)[0])
        mixtures.append(audio_transforms(fg, bg)[0])
        labels.append(label)

    cues = torch.stack(cues)
    foregrounds = torch.stack(foregrounds)
    backgrounds = torch.stack(backgrounds)
    mixtures = torch.stack(mixtures)
    labels = torch.tensor(labels).type(torch.LongTensor)
    return cues, foregrounds, backgrounds, mixtures, labels

dataloader = torch.utils.data.DataLoader(
            dataset,
            batch_size=1,
            num_workers=1,
            collate_fn=collate_fn
        )

1


/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/torchaudio/functional/functional.py:1371: UserWarning: "kaiser_window" resampling method name is being deprecated and replaced by "sinc_interp_kaiser" in the next release. The default behavior remains unchanged.
  warnings.warn(


In [21]:
conv_modules = {name:module for name, module in model.model._orig_mod.model_dict.named_children() if 'conv' in name or 'relu' in name}
# add relu fc 
relu_fc = model.model._orig_mod.relufc
modules = {**conv_modules, **{'relufc': relu_fc}}

In [22]:
modules

{'conv_block_0': Sequential(
   (0): LayerNorm((2, 40, 20000), eps=1e-05, elementwise_affine=True)
   (1): Conv2dSame(2, 32, kernel_size=(2, 34), stride=(1, 1), bias=False)
   (2): ReLU()
   (3): HannPooling2d()
 ),
 'conv_block_1': Sequential(
   (0): LayerNorm((32, 20, 5000), eps=1e-05, elementwise_affine=True)
   (1): Conv2dSame(32, 64, kernel_size=(2, 14), stride=(1, 1), bias=False)
   (2): ReLU()
   (3): HannPooling2d()
 ),
 'conv_block_2': Sequential(
   (0): LayerNorm((64, 10, 1250), eps=1e-05, elementwise_affine=True)
   (1): Conv2dSame(64, 256, kernel_size=(5, 5), stride=(1, 1), bias=False)
   (2): ReLU()
   (3): HannPooling2d()
 ),
 'conv_block_3': Sequential(
   (0): LayerNorm((256, 10, 250), eps=1e-05, elementwise_affine=True)
   (1): Conv2dSame(256, 512, kernel_size=(5, 5), stride=(1, 1), bias=False)
   (2): ReLU()
   (3): HannPooling2d()
 ),
 'conv_block_4': Sequential(
   (0): LayerNorm((512, 10, 63), eps=1e-05, elementwise_affine=True)
   (1): Conv2dSame(512, 512, kerne

In [23]:
# init array to store activations
activations = {}

def get_activation(name):
    def hook(model, input, output):
        activations[name] = output.detach()
    return hook

# dicts to store activations
fg_reps = {}
bg_reps = {}
mixture_reps = {}

# register hooks 
for name, module in modules.items():
    print(name)
    if 'relufc' in name:
        module.register_forward_hook(get_activation(name)) 
    else:
        module[2].register_forward_hook(get_activation(name)) # [2] is relu 
    # lists to save acts in per-layer
    fg_reps[name] = []
    bg_reps[name] = []
    mixture_reps[name] = []

# send model to gpu 
model = model.eval().cuda()


conv_block_0
conv_block_1
conv_block_2
conv_block_3
conv_block_4
conv_block_5
conv_block_6
relufc


In [36]:
# add cochleagram to dicts 
mixture_reps['cochleagram'] = []
fg_reps['cochleagram'] = []
bg_reps['cochleagram'] = []

n_activations = 10
# get activations 
with torch.no_grad():
    for ix, batch in tqdm(enumerate(dataloader),  total = n_activations):
        fg_cue, foreground, background, mixture, fg_target = batch

        # send to device
        foreground, background, mixture = foreground.cuda(), background.cuda(), mixture.cuda()
        fg_cue =  fg_cue.cuda()

        # convert to cochleagrams
        fg_cue, mixture = coch_gram(fg_cue, mixture)
        foreground, background = coch_gram(foreground, background)

        # save inputs
        mixture_reps['cochleagram'].append(mixture.view(1,-1).cpu())
        fg_reps['cochleagram'].append(foreground.view(1,-1).cpu())
        bg_reps['cochleagram'].append(background.view(1,-1).cpu())


        # run mixture
        model(fg_cue, mixture, None)
            
        for layer in mixture_reps.keys():
            if layer == 'cochleagram':
                continue 
            mixture_reps[layer].append(activations[layer].view(1,-1).cpu())
                
        # run fg
        model(fg_cue, foreground, None)
            
        for layer in fg_reps.keys():
            if layer == 'cochleagram':
                continue 
            fg_reps[layer].append(activations[layer].view(1,-1).cpu())
                
        # run bg
        model(fg_cue, background, None)
            
        for layer in bg_reps.keys():
            if layer == 'cochleagram':
                continue 
            bg_reps[layer].append(activations[layer].view(1,-1).cpu())
            
        if ix == n_activations:
            break
    

# concat and store as numpy arrays
mixture_reps = {layer:torch.concat(acts,axis=0).numpy().astype('float16')
    for layer,acts in mixture_reps.items()}

fg_reps = {layer:torch.concat(acts,axis=0).numpy().astype('float16')
                for layer,acts in fg_reps.items()}

bg_reps = {layer:torch.concat(acts,axis=0).numpy().astype('float16')
                for layer,acts in bg_reps.items()}

  0%|          | 0/10 [00:12<?, ?it/s]


BackendCompilerFailed: backend='inductor' raised:
RuntimeError: Found NVIDIA GeForce GTX 1080 Ti which is too old to be supported by the triton GPU compiler, which is used as the backend. Triton only supports devices of CUDA Capability >= 7.0, but your device is of CUDA capability 6.1

Set TORCH_LOGS="+dynamo" and TORCHDYNAMO_VERBOSE=1 for more information


You can suppress this exception and fall back to eager by setting:
    import torch._dynamo
    torch._dynamo.config.suppress_errors = True
